In [1]:
import numpy as np                      # cálculo numérico (vectores, matrices)
import pandas as pd                     # manejo de tablas (leer/escribir CSV/TSV)
from itertools import product           # para generar todas las combinaciones de k-mers
from collections import Counter         # para contar cuántas veces aparece cada k-mer
from sklearn.ensemble import RandomForestClassifier      # el modelo (Random Forest)
from sklearn.model_selection import StratifiedGroupKFold # la validación cruzada
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score  # métricas

INTERIM = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/data/interim"  # carpeta de datos
VENTANA = 500                           # tamaño fijo de cada fragmento (500 pb)
K = 4                                  # tamaño del k-mer (subcadenas de 4 letras)
KMERS = ["".join(p) for p in product("ACGT", repeat=K)]  # las 256 combinaciones: AAAA..TTTT
KMER_IDX = {km: i for i, km in enumerate(KMERS)}         # diccionario k-mer -> posición en el vector
print("Listo. k-mers posibles:", len(KMERS))             # debe imprimir 256

Listo. k-mers posibles: 256


In [2]:
#Fragmentación con relleno de ceros

def fragmentar(seq, ventana=VENTANA):
    seq = seq.upper()                   # pasa todo a mayúsculas (ACGT)
    L = len(seq)                        # longitud del gen en nucleótidos
    frags = []                          # lista donde guardaremos los fragmentos

    if L < ventana:                     # CASO 1: gen más corto que 500
        frags.append(seq + "0" * (ventana - L))   # -----------se rellena con ceros a la derecha hasta 500

    else:                               # CASO 2: gen de 500 o más
        n = L // ventana                # cuántas ventanas completas de 500 caben
        for i in range(n):              # recorre cada ventana completa
            
            frags.append(seq[i*ventana:(i+1)*ventana])  # --------------corta el bloque de 500 (posición i)
        resto = L - n * ventana         # cuántos nucleótidos sobran al final
        if resto > 0:                   # si sobró un pedazo menor a 500...
            frags.append(seq[n*ventana:] + "0" * (ventana - resto))  # ----------se rellena con ceros hasta 500
    return frags                        # devuelve la lista de fragmentos (todos de 500)

for s in ["A"*400, "C"*1300, "G"*650]:  # prueba con 3 genes de distinta longitud
    print(len(s), "->", [len(f) for f in fragmentar(s)])  # confirma que todos midan 500

400 -> [500]
1300 -> [500, 500, 500]
650 -> [500, 500]


In [3]:
#Leer FASTA y generar el dataset fragmentado

def leer_fasta(path):
    seqs, header, chunks = {}, None, []     # dict de secuencias, encabezado actual, trozos de línea
    with open(path) as f:                   # abre el archivo FASTA
        for line in f:                      # recorre línea por línea
            line = line.rstrip("\n")        # quita el salto de línea
            if line.startswith(">"):        # si es un encabezado (empieza con >)
                if header: seqs[header] = "".join(chunks)  # guarda la secuencia anterior completa
                header = line[1:].split()[0]               # nuevo id (sin > y hasta el 1er espacio)
                chunks = []                                # reinicia el acumulador
            else:
                chunks.append(line.strip()) # va juntando las líneas de secuencia
        if header: seqs[header] = "".join(chunks)  # guarda la última secuencia
    return seqs                             # devuelve {id: secuencia}

positivas = leer_fasta(f"{INTERIM}/combinado_v4_reducido.fasta")  # genes de resistencia (positivos)
negativas = leer_fasta(f"{INTERIM}/grupo_negativo_v5.fasta")      # genes normales (negativos)

filas = []                                  # aquí van todos los fragmentos etiquetados
for seqs, etiqueta in [(positivas, 1), (negativas, 0)]:  # 1=resistente, 0=no resistente
    for header, seq in seqs.items():        # recorre cada gen
        for idx, frag in enumerate(fragmentar(seq)):     # fragmenta y numera cada pedazo
            filas.append((f"{header}__frag{idx}", frag, etiqueta))  # (id_fragmento, secuencia, etiqueta)

with open(f"{INTERIM}/dataset_fragmentado_500.tsv", "w") as fout:  # guarda el resultado
    fout.write("id\tlectura\tetiqueta\n")   # encabezado del archivo
    for fid, frag, et in filas:
        fout.write(f"{fid}\t{frag}\t{et}\n") # una fila por fragmento

n_pos = sum(1 for _,_,e in filas if e==1)   # cuenta positivos
print(f"Total fragmentos: {len(filas)} | Positivos: {n_pos} | Negativos: {len(filas)-n_pos}")

Total fragmentos: 72874 | Positivos: 9459 | Negativos: 63415


In [4]:
#Función k-mer y generar features

def vector_kmer(seq):
    seq = seq.upper()                       # a mayúsculas
    counts = Counter(seq[i:i+K] for i in range(len(seq)-K+1))  # cuenta cada subcadena de 4 letras
    total = sum(c for km, c in counts.items() if km in KMER_IDX)  # suma SOLO k-mers válidos (ACGT)
    vec = [0.0] * len(KMERS)                # vector de 256 ceros
    if total > 0:                           # evita dividir entre cero
        for km, c in counts.items():        # recorre cada k-mer contado
            if km in KMER_IDX:              # si es un k-mer válido (ignora los que tienen '0')
                vec[KMER_IDX[km]] = c / total  # frecuencia relativa = conteo / total
    return vec                              # devuelve el vector de 256 frecuencias

df_frag = pd.read_csv(f"{INTERIM}/dataset_fragmentado_500.tsv", sep="\t")  # carga los fragmentos
matriz = np.array([vector_kmer(s) for s in df_frag["lectura"]])  # convierte cada fragmento en vector
feat = pd.DataFrame(matriz, columns=KMERS)          # matriz -> tabla con 256 columnas (una por k-mer)
feat.insert(0, "etiqueta", df_frag["etiqueta"].values)  # agrega la columna etiqueta al inicio
feat.insert(0, "id", df_frag["id"].values)              # agrega la columna id al inicio
feat.to_csv(f"{INTERIM}/features_kmer4_fragmentado.csv", index=False)  # guarda las features

print("Features generadas:", feat.shape)    # (nº fragmentos, 258) = id + etiqueta + 256 k-mers
feat.head()                                 # muestra las primeras filas

Features generadas: (72874, 258)


,id,etiqueta,AAAA,AAAC,AAAG,AAAT,AACA,AACC,AACG,AACT,...,TTCG,TTCT,TTGA,TTGC,TTGG,TTGT,TTTA,TTTC,TTTG,TTTT
0,CARD|gb|AF028812.1|+|392-887|ARO:3002867|dfrF_...,1,0.010163,0.002033,0.010163,0.016260,0.010163,0.000000,0.002033,0.000000,...,0.000000,0.002033,0.012195,0.006098,0.006098,0.006098,0.012195,0.006098,0.012195,0.008130
1,CARD|gb|AY234334.1|+|0-846|ARO:3000600|Erm(34)...,1,0.034205,0.014085,0.010060,0.008048,0.008048,0.004024,0.012072,0.006036,...,0.006036,0.004024,0.006036,0.002012,0.004024,0.010060,0.010060,0.004024,0.002012,0.008048
2,CARD|gb|AY234334.1|+|0-846|ARO:3000600|Erm(34)...,1,0.014577,0.008746,0.020408,0.000000,0.005831,0.005831,0.000000,0.000000,...,0.005831,0.002915,0.002915,0.000000,0.002915,0.008746,0.005831,0.008746,0.002915,0.020408
3,CARD|gb|EU408351.1|+|16572-17454|ARO:3002511|L...,1,0.016097,0.002012,0.012072,0.012072,0.008048,0.006036,0.002012,0.002012,...,0.002012,0.004024,0.002012,0.006036,0.002012,0.006036,0.008048,0.004024,0.006036,0.002012
4,CARD|gb|EU408351.1|+|16572-17454|ARO:3002511|L...,1,0.034301,0.002639,0.010554,0.010554,0.005277,0.000000,0.002639,0.000000,...,0.000000,0.002639,0.013193,0.007916,0.005277,0.000000,0.013193,0.002639,0.007916,0.013193


In [5]:
#Preparar X, y, grupos

df = pd.read_csv(f"{INTERIM}/features_kmer4_fragmentado.csv")  # carga las features
df["gen_origen"] = df["id"].str.split("__frag").str[0]        # extrae el gen de origen de cada fragmento

X = df.drop(columns=["id", "etiqueta", "gen_origen"]).values  # X = solo las 256 columnas de k-mers
y = df["etiqueta"].values                                     # y = la etiqueta (0/1) que queremos predecir
grupos = df["gen_origen"].values                             # grupo = gen de origen (para no mezclar)

print(f"X: {X.shape} | Positivos: {y.sum()} | Negativos: {(y==0).sum()}")
print(f"Genes únicos: {df['gen_origen'].nunique()}")

X: (72874, 256) | Positivos: 9459 | Negativos: 63415
Genes únicos: 30741


/tmp/ipykernel_1374/4176524660.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["gen_origen"] = df["id"].str.split("__frag").str[0]        # extrae el gen de origen de cada fragmento


In [6]:
#Balanceo + validación cruzada (cross validation)

skf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)  # 10 particiones (10% val c/u)
rng = np.random.default_rng(42)             # generador aleatorio con semilla fija (reproducible)
recs, precs, f1s, aucs = [], [], [], []     # listas para guardar la métrica de cada partición

for i, (tr, va) in enumerate(skf.split(X, y, grupos), 1):  # recorre las 10 rondas
    Xtr, ytr = X[tr], y[tr]                 # datos de ENTRENAMIENTO de esta ronda
    
    # --- BALANCEO (solo en entrenamiento) ---
    pos = np.where(ytr == 1)[0]             # posiciones de los positivos
    neg = np.where(ytr == 0)[0]             # posiciones de los negativos
    neg_ds = rng.choice(neg, size=len(pos), replace=False)  # elige tantos neg como pos (sin repetir)
    bal = np.concatenate([pos, neg_ds])     # junta positivos + negativos submuestreados (50/50)
    Xb, yb = Xtr[bal], ytr[bal]             # conjunto de entrenamiento ya balanceado

    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)  # 200 árboles
    rf.fit(Xb, yb)                          # entrena el modelo con los datos balanceados
    yp = rf.predict(X[va])                  # predice etiquetas 0/1 en la VALIDACIÓN (sin balancear)
    pr = rf.predict_proba(X[va])[:, 1]      # probabilidad de ser resistente (para el AUC)

    r = recall_score(y[va], yp)             # recall de esta ronda
    p = precision_score(y[va], yp, zero_division=0)  # precisión de esta ronda
    recs.append(r); precs.append(p)         # guarda recall y precisión
    f1s.append(f1_score(y[va], yp)); aucs.append(roc_auc_score(y[va], pr))  # guarda F1 y AUC
    print(f"Fold {i:>2}: recall={r:.3f}  precision={p:.3f}  F1={f1s[-1]:.3f}  AUC={aucs[-1]:.3f}")

print("\n=== PROMEDIOS (10-fold) ===")
print(f"Recall:    {np.mean(recs):.3f} +/- {np.std(recs):.3f}")   # promedio ± desviación
print(f"Precision: {np.mean(precs):.3f} +/- {np.std(precs):.3f}")
print(f"F1:        {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}")
print(f"AUC-ROC:   {np.mean(aucs):.3f} +/- {np.std(aucs):.3f}")

Fold  1: recall=0.752  precision=0.418  F1=0.538  AUC=0.874
Fold  2: recall=0.762  precision=0.404  F1=0.528  AUC=0.884
Fold  3: recall=0.776  precision=0.411  F1=0.537  AUC=0.883
Fold  4: recall=0.779  precision=0.427  F1=0.552  AUC=0.890
Fold  5: recall=0.758  precision=0.404  F1=0.527  AUC=0.867
Fold  6: recall=0.733  precision=0.412  F1=0.527  AUC=0.872
Fold  7: recall=0.780  precision=0.405  F1=0.534  AUC=0.882
Fold  8: recall=0.799  precision=0.416  F1=0.547  AUC=0.901
Fold  9: recall=0.796  precision=0.420  F1=0.550  AUC=0.900
Fold 10: recall=0.792  precision=0.422  F1=0.551  AUC=0.886

=== PROMEDIOS (10-fold) ===
Recall:    0.773 +/- 0.020
Precision: 0.414 +/- 0.008
F1:        0.539 +/- 0.010
AUC-ROC:   0.884 +/- 0.011


In [7]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (recall_score, precision_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

INTERIM = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/data/interim"

# --- Cargar datos y preparar X, y, grupos ---
df = pd.read_csv(f"{INTERIM}/features_kmer4_fragmentado.csv")
df["gen_origen"] = df["id"].str.split("__frag").str[0]
X = df.drop(columns=["id", "etiqueta", "gen_origen"]).values
y = df["etiqueta"].values
grupos = df["gen_origen"].values

# --- Validación cruzada con balanceo ---
skf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
rng = np.random.default_rng(42)
recs, precs, f1s, aucs = [], [], [], []
y_true_all, y_pred_all, y_prob_all = [], [], []

for i, (tr, va) in enumerate(skf.split(X, y, grupos), 1):
    Xtr, ytr = X[tr], y[tr]
    pos = np.where(ytr == 1)[0]
    neg = np.where(ytr == 0)[0]
    neg_ds = rng.choice(neg, size=len(pos), replace=False)
    bal = np.concatenate([pos, neg_ds])
    Xb, yb = Xtr[bal], ytr[bal]

    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
    rf.fit(Xb, yb)
    yp = rf.predict(X[va])
    pr = rf.predict_proba(X[va])[:, 1]

    recs.append(recall_score(y[va], yp))
    precs.append(precision_score(y[va], yp, zero_division=0))
    f1s.append(f1_score(y[va], yp))
    aucs.append(roc_auc_score(y[va], pr))
    y_true_all.extend(y[va]); y_pred_all.extend(yp); y_prob_all.extend(pr)


# --- Promedios ---
print("=== PROMEDIOS (10-fold) ===")
print(f"Recall:    {np.mean(recs):.3f} +/- {np.std(recs):.3f}")
print(f"Precision: {np.mean(precs):.3f} +/- {np.std(precs):.3f}")
print(f"F1:        {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}")
print(f"AUC-ROC:   {np.mean(aucs):.3f} +/- {np.std(aucs):.3f}\n")



# --- Matriz de confusión + reporte + AUC (10 particiones juntas) ---
y_true_all = np.array(y_true_all); y_pred_all = np.array(y_pred_all); y_prob_all = np.array(y_prob_all)
cm = confusion_matrix(y_true_all, y_pred_all)
print("Matriz de confusión:")
print(f"                  Predijo NEG   Predijo POS")
print(f"  Real NEG (0)      {cm[0,0]:>8}      {cm[0,1]:>8}")
print(f"  Real POS (1)      {cm[1,0]:>8}      {cm[1,1]:>8}")
print()
print(classification_report(y_true_all, y_pred_all, target_names=["no_resist (0)", "resist (1)"], digits=3))
print(f"AUC-ROC: {roc_auc_score(y_true_all, y_prob_all):.3f}")

/tmp/ipykernel_1374/1907756883.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["gen_origen"] = df["id"].str.split("__frag").str[0]


=== PROMEDIOS (10-fold) ===
Recall:    0.773 +/- 0.020
Precision: 0.414 +/- 0.008
F1:        0.539 +/- 0.010
AUC-ROC:   0.884 +/- 0.011

Matriz de confusión:
                  Predijo NEG   Predijo POS
  Real NEG (0)         53073         10342
  Real POS (1)          2151          7308

               precision    recall  f1-score   support

no_resist (0)      0.961     0.837     0.895     63415
   resist (1)      0.414     0.773     0.539      9459

     accuracy                          0.829     72874
    macro avg      0.688     0.805     0.717     72874
 weighted avg      0.890     0.829     0.849     72874

AUC-ROC: 0.884


In [ ]:
import numpy as np, pandas as pd, gc
from itertools import product
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

INTERIM = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/data/interim"
dfrag = pd.read_csv(f"{INTERIM}/dataset_fragmentado_500.tsv", sep="\t")
dfrag["gen"] = dfrag["id"].str.split("__frag").str[0]
y = dfrag["etiqueta"].values
g = dfrag["gen"].values

def vector_kmer(seq, K, idx):
    c = Counter(seq[i:i+K] for i in range(len(seq)-K+1))
    tot = sum(v for k, v in c.items() if k in idx)
    out = [0.0]*len(idx)
    if tot > 0:
        for k, v in c.items():
            if k in idx: out[idx[k]] = v/tot
    return out

for K in [4, 5, 6]:
    print(f"\n{'='*55}\n  K = {K}\n{'='*55}")
    kl = ["".join(p) for p in product("ACGT", repeat=K)]
    idx = {k: i for i, k in enumerate(kl)}
    X = np.array([vector_kmer(s, K, idx) for s in dfrag["lectura"]], dtype=np.float32)

    oof_l = np.full(len(y), -1); oof_p = np.zeros(len(y))
    skf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
    rng = np.random.default_rng(42)
    for tr, va in skf.split(X, y, g):
        p = np.where(y[tr]==1)[0]; n = np.where(y[tr]==0)[0]
        bal = np.concatenate([p, rng.choice(n, size=len(p), replace=False)])
        rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
        rf.fit(X[tr][bal], y[tr][bal])
        oof_l[va] = rf.predict(X[va]); oof_p[va] = rf.predict_proba(X[va])[:,1]

    print(f"Features: {X.shape[1]}")
    print(classification_report(y, oof_l, target_names=["no_resist","resist"], digits=3))
    print(f"AUC-ROC:  {roc_auc_score(y, oof_p):.3f}")
    print(f"Accuracy: {accuracy_score(y, oof_l):.3f}")
    del X; gc.collect()


  K = 4
Features: 256
              precision    recall  f1-score   support

   no_resist      0.961     0.837     0.895     63415
      resist      0.414     0.773     0.539      9459

    accuracy                          0.829     72874
   macro avg      0.688     0.805     0.717     72874
weighted avg      0.890     0.829     0.849     72874

AUC-ROC:  0.884
Accuracy: 0.829

  K = 5
Features: 1024
              precision    recall  f1-score   support

   no_resist      0.961     0.850     0.902     63415
      resist      0.432     0.766     0.553      9459

    accuracy                          0.839     72874
   macro avg      0.696     0.808     0.727     72874
weighted avg      0.892     0.839     0.856     72874

AUC-ROC:  0.886
Accuracy: 0.839

  K = 6
Features: 4096
              precision    recall  f1-score   support

   no_resist      0.958     0.865     0.910     63415
      resist      0.453     0.749     0.565      9459

    accuracy                          0.850    

In [ ]:
import numpy as np, pandas as pd, gc
from itertools import product
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

INTERIM = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/data/interim"
dfrag = pd.read_csv(f"{INTERIM}/dataset_fragmentado_500.tsv", sep="\t")
dfrag["gen"] = dfrag["id"].str.split("__frag").str[0]
y = dfrag["etiqueta"].values; g = dfrag["gen"].values

K = 6
kl = ["".join(p) for p in product("ACGT", repeat=K)]
idx = {k: i for i, k in enumerate(kl)}
def vk(seq):
    c = Counter(seq[i:i+K] for i in range(len(seq)-K+1))
    tot = sum(v for k, v in c.items() if k in idx); out=[0.0]*len(idx)
    if tot>0:
        for k,v in c.items():
            if k in idx: out[idx[k]] = v/tot
    return out
X = np.array([vk(s) for s in dfrag["lectura"]], dtype=np.float32)

oof_l = np.full(len(y),-1); oof_p = np.zeros(len(y))
skf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
rng = np.random.default_rng(42)
for tr, va in skf.split(X, y, g):
    p=np.where(y[tr]==1)[0]; n=np.where(y[tr]==0)[0]
    bal=np.concatenate([p, rng.choice(n, size=len(p), replace=False)])
    rf=RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
    rf.fit(X[tr][bal], y[tr][bal]); oof_l[va]=rf.predict(X[va]); oof_p[va]=rf.predict_proba(X[va])[:,1]

print("K=6, features:", X.shape[1])
print(classification_report(y, oof_l, target_names=["no_resist","resist"], digits=3))
print(f"AUC-ROC:  {roc_auc_score(y, oof_p):.3f}")
print(f"Accuracy: {accuracy_score(y, oof_l):.3f}")

K=6, features: 4096
              precision    recall  f1-score   support

   no_resist      0.958     0.865     0.910     63415
      resist      0.453     0.749     0.565      9459

    accuracy                          0.850     72874
   macro avg      0.706     0.807     0.737     72874
weighted avg      0.893     0.850     0.865     72874

AUC-ROC:  0.884
Accuracy: 0.850


In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

INTERIM = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/data/interim"
RUTA_APP = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/app"

# 1. Cargar las features de K=4 (72,874 fragmentos x 256 k-mers)
df = pd.read_csv(f"{INTERIM}/features_kmer4_fragmentado.csv")
X = df.drop(columns=["id", "etiqueta"]).values
y = df["etiqueta"].values

# 2. Balanceo 50/50 en todo el conjunto de entrenamiento
rng = np.random.default_rng(42)
pos = np.where(y == 1)[0]
neg = np.where(y == 0)[0]
neg_ds = rng.choice(neg, size=len(pos), replace=False)
bal = np.concatenate([pos, neg_ds])

X_bal, y_bal = X[bal], y[bal]

# 3. Entrenar el modelo definitivo
print(f"Entrenando modelo final con {len(bal)} fragmentos balanceados...")
rf_final = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
rf_final.fit(X_bal, y_bal)

# 4. Guardar el modelo en la carpeta 'app' para que la GUI lo encuentre directo
os.makedirs(RUTA_APP, exist_ok=True)
ruta_modelo = os.path.join(RUTA_APP, "modelo_rf_k4.joblib")
joblib.dump(rf_final, ruta_modelo)
print(f"¡Modelo guardado con éxito en: {ruta_modelo}!")

Entrenando modelo final con 18918 fragmentos balanceados...
¡Modelo guardado con éxito en: /mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/app/modelo_rf_k4.joblib!


In [1]:
# ============ CELDA: dimensiones y porcentajes (id95 e id90) ============
# No entrena nada y no carga ningún modelo: solo reproduce el conteo de datos
# con la misma lógica y la misma semilla que los cuadernos, para los dos sets.

import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

INTERIM = Path("/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/data/interim")
K, VENTANA, SEED, FRAC_TEST = 6, 500, 42, 0.10

CONFIGS = [("id95", None),
           ("id90", INTERIM / "positivo_id90_limpio.fasta")]

# ---------- se lee una sola vez ----------
frag = pd.read_csv(INTERIM / "dataset_fragmentado_500.tsv", sep="\t")
col_id, col_seq, col_lab = "id", "lectura", "etiqueta"
frag[col_lab] = frag[col_lab].astype(int)
frag["gen"] = frag[col_id].str.replace(r"__frag\d+$", "", regex=True)

lab1k = pd.read_csv(INTERIM / "muestra_genes_labels.tsv", sep="\t")
ids_1k = set(lab1k.iloc[:, 0].astype(str))
genes_frag = set(frag["gen"])
excluir = ids_1k & genes_frag
faltan = ids_1k - excluir
if faltan:
    variantes = {}
    for g in genes_frag:
        variantes.setdefault(g.split("|", 1)[-1], g)
    for f in list(faltan):
        if f in variantes:
            excluir.add(variantes[f]); faltan.discard(f)
assert not faltan, f"no encontré {len(faltan)} genes del 1k+1k"


def pct(a, b):
    return f"{100*a/b:5.1f} %" if b else "   -  "


def bloque(etiqueta, filtro):
    print("=" * 74)
    print(f"  {etiqueta}" + (f"   (positivos filtrados: {filtro.name})" if filtro else
                             "   (positivos sin filtrar)"))
    print("=" * 74)

    # ---------- 1) dataset completo ----------
    n_tot = len(frag)
    np_, nn = int((frag[col_lab] == 1).sum()), int((frag[col_lab] == 0).sum())
    gp = frag.loc[frag[col_lab] == 1, "gen"].nunique()
    gn = frag.loc[frag[col_lab] == 0, "gen"].nunique()
    print(f"\n1) DATASET COMPLETO")
    print(f"   fragmentos : {n_tot:>8,}            genes: {frag['gen'].nunique():>7,}")
    print(f"   positivos  : {np_:>8,}  {pct(np_, n_tot)}   de {gp:>7,} genes")
    print(f"   negativos  : {nn:>8,}  {pct(nn, n_tot)}   de {gn:>7,} genes")
    print(f"   proporción natural: {nn/max(1,np_):.1f} negativos por cada positivo")

    # ---------- 2) quitar el 1k+1k ----------
    pool = frag[~frag["gen"].isin(excluir)].reset_index(drop=True)
    print(f"\n2) TRAS EXCLUIR LOS 2000 GENES DEL 1k+1k")
    print(f"   fragmentos excluidos: {n_tot - len(pool):>8,}")
    print(f"   quedan              : {len(pool):>8,}")

    # ---------- 3) filtro de identidad ----------
    if filtro is not None:
        hdrs = [l[1:].strip() for l in open(filtro, encoding="utf-8", errors="replace")
                if l.startswith(">")]
        ids_ok = set(hdrs) | {h.split()[0] for h in hdrs}
        quitar = (pool[col_lab] == 1) & (~pool["gen"].isin(ids_ok))
        print(f"\n3) FILTRO DE IDENTIDAD")
        print(f"   secuencias en el FASTA filtrado        : {len(hdrs):>8,}")
        print(f"   fragmentos positivos que se quitan     : {int(quitar.sum()):>8,}")
        pool = pool[~quitar].reset_index(drop=True)
    else:
        print(f"\n3) FILTRO DE IDENTIDAD: ninguno (set de 95 %)")

    np_, nn = int((pool[col_lab] == 1).sum()), int((pool[col_lab] == 0).sum())
    print(f"\n   POOL final: {len(pool):>8,} fragmentos")
    print(f"     positivos: {np_:>8,}  {pct(np_, len(pool))}   "
          f"de {pool.loc[pool[col_lab]==1,'gen'].nunique():>7,} genes")
    print(f"     negativos: {nn:>8,}  {pct(nn, len(pool))}   "
          f"de {pool.loc[pool[col_lab]==0,'gen'].nunique():>7,} genes")

    # ---------- 4) corte por gen ----------
    y, g = pool[col_lab].values, pool["gen"].values
    sgkf = StratifiedGroupKFold(n_splits=int(round(1/FRAC_TEST)), shuffle=True, random_state=SEED)
    tr, te = next(sgkf.split(np.zeros(len(y)), y, g))
    assert not set(g[tr]) & set(g[te]), "FUGA entre train y test"
    print(f"\n4) CORTE POR GEN (StratifiedGroupKFold {int(round(1/FRAC_TEST))}, semilla {SEED})")
    for nom, idx in [("train", tr), ("test ", te)]:
        p, n = int((y[idx] == 1).sum()), int((y[idx] == 0).sum())
        print(f"   {nom}: {len(idx):>7,} frags  ({len(set(g[idx])):>6,} genes)   "
              f"pos={p:>6,} {pct(p,len(idx))}   neg={n:>6,} {pct(n,len(idx))}")

    # ---------- 5) undersampling ----------
    rng = np.random.default_rng(SEED)
    pos = tr[y[tr] == 1]
    neg = tr[y[tr] == 0]
    neg_ds = rng.choice(neg, size=len(pos), replace=False)
    tr_bal = np.concatenate([pos, neg_ds])
    print(f"\n5) RANDOM UNDERSAMPLING (solo en train, solo negativos)")
    print(f"   antes : pos={len(pos):>6,}  neg={len(neg):>6,}   (razón {len(neg)/max(1,len(pos)):.1f}:1)")
    print(f"   después: pos={len(pos):>6,}  neg={len(neg_ds):>6,}   (razón 1.0:1)")
    print(f"   negativos descartados: {len(neg)-len(neg_ds):>6,} de {len(neg):,}  "
          f"({100*(len(neg)-len(neg_ds))/max(1,len(neg)):.1f} %)")

    # ---------- 6) shapes ----------
    d = 4 ** K
    gb = lambda n: n * d * 4 / 1e9
    print(f"\n6) DIMENSIONES QUE VE EL MODELO   (K={K} -> {d:,} columnas)")
    print(f"   X_train.shape = ({len(tr_bal):>7,}, {d:,})    ~{gb(len(tr_bal)):.2f} GB en float32")
    print(f"   X_test.shape  = ({len(te):>7,}, {d:,})    ~{gb(len(te)):.2f} GB en float32")
    print(f"   y_train: 50.0 % positivos / 50.0 % negativos (balanceado)")
    p_te = int((y[te] == 1).sum())
    print(f"   y_test : {pct(p_te,len(te)).strip()} positivos (proporción natural, sin balancear)")

    return dict(config=etiqueta, pool=len(pool), pool_pos=np_, pool_neg=nn,
                train_frags=len(tr_bal), train_pos=len(pos),
                neg_disp=len(neg), neg_usados=len(neg_ds),
                test_frags=len(te), test_pos=p_te)


resumen = [bloque(e, f) for e, f in CONFIGS]

# ---------- comparación ----------
print("\n" + "=" * 74)
print("  COMPARACIÓN")
print("=" * 74)
t = pd.DataFrame(resumen).set_index("config")
t["% neg descartados"] = (100 * (t.neg_disp - t.neg_usados) / t.neg_disp).round(1)
t["% pos del pool"] = (100 * t.pool_pos / t.pool).round(1)
print(t[["pool", "pool_pos", "pool_neg", "% pos del pool",
         "train_frags", "train_pos", "neg_disp", "neg_usados",
         "% neg descartados", "test_frags", "test_pos"]].T.to_string())
print("\nLectura: 'neg_disp' son los negativos disponibles en train y 'neg_usados' los que")
print("de verdad entrenaron. La diferencia es lo que el undersampling tira en cada corrida.")

  id95   (positivos sin filtrar)

1) DATASET COMPLETO
   fragmentos :   72,874            genes:  30,741
   positivos  :    9,459   13.0 %   de   4,228 genes
   negativos  :   63,415   87.0 %   de  26,513 genes
   proporción natural: 6.7 negativos por cada positivo

2) TRAS EXCLUIR LOS 2000 GENES DEL 1k+1k
   fragmentos excluidos:    4,667
   quedan              :   68,207

3) FILTRO DE IDENTIDAD: ninguno (set de 95 %)

   POOL final:   68,207 fragmentos
     positivos:    7,229   10.6 %   de   3,228 genes
     negativos:   60,978   89.4 %   de  25,513 genes

4) CORTE POR GEN (StratifiedGroupKFold 10, semilla 42)
   train:  61,386 frags  (25,867 genes)   pos= 6,506  10.6 %   neg=54,880  89.4 %
   test :   6,821 frags  ( 2,874 genes)   pos=   723  10.6 %   neg= 6,098  89.4 %

5) RANDOM UNDERSAMPLING (solo en train, solo negativos)
   antes : pos= 6,506  neg=54,880   (razón 8.4:1)
   después: pos= 6,506  neg= 6,506   (razón 1.0:1)
   negativos descartados: 48,374 de 54,880  (88.1 %)

6) 